## Qwen3-VL-8B Fold 0 24-class LoRA 학습

이 노트북은 팀 공통 `fold_assignments.csv`를 그대로 읽어 Fold 0만 Validation으로 고정합니다. Fold를 다시 계산하지 않으며 Test 데이터에는 접근하지 않습니다.

현재 최고 `fold0_dist_aug_v1/final_adapter`를 초기값으로 사용합니다. 네 이미지의 가능한 시간순서 24개를 하나의 클래스로 정의하고, Qwen의 멀티모달 표현 위에 단일 Linear classification head를 학습합니다.

기존 문자열 생성 대신 24-class cross-entropy로 전체 순열을 직접 최적화합니다. Qwen 백본 하나와 LoRA·분류 헤드 하나만 사용하며 Test 데이터에는 접근하지 않습니다.

### 1. A100, Google Drive, 공통 Fold 파일 확인

학습 전에 GPU와 필수 경로만 검사합니다. 이 셀은 데이터를 학습하거나 모델을 다운로드하지 않습니다. `fold_assignments.csv`는 앞 단계에서 Colab의 scikit-learn 1.6.1로 생성한 파일이어야 합니다.

In [ ]:
from pathlib import Path
import torch
from google.colab import drive

assert torch.cuda.is_available(), 'GPU 런타임이 아닙니다.'
assert torch.cuda.is_bf16_supported(), 'BF16을 지원하는 GPU가 필요합니다.'
drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/SNU')
DATA_DIR = PROJECT_DIR / 'data'
TRAIN_CSV = DATA_DIR / 'train.csv'
TRAIN_IMAGE_DIR = DATA_DIR / 'train'
FOLD_CSV = PROJECT_DIR / 'outputs' / 'splits' / 'fold_assignments.csv'

MODEL_NAME = 'Qwen/Qwen3-VL-8B-Instruct'
SOURCE_RUN_NAME = 'fold0_dist_aug_v1'
SOURCE_ADAPTER_DIR = PROJECT_DIR / 'outputs' / 'qwen3_vl_8b_lora' / SOURCE_RUN_NAME / 'final_adapter'
RUN_NAME = 'fold0_cls24_v1'
RUN_DIR = PROJECT_DIR / 'outputs' / 'qwen3_vl_8b_lora' / RUN_NAME
FINAL_ADAPTER_DIR = RUN_DIR / 'final_adapter'
CLASSIFIER_PATH = FINAL_ADAPTER_DIR / 'permutation_classifier.safetensors'
CLASS_MAPPING_PATH = FINAL_ADAPTER_DIR / 'class_mapping.json'

for required_path in (TRAIN_CSV, TRAIN_IMAGE_DIR, FOLD_CSV, SOURCE_ADAPTER_DIR):
    assert required_path.exists(), f'필수 경로가 없습니다: {required_path}'
RUN_DIR.mkdir(parents=True, exist_ok=True)

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'프로젝트: {PROJECT_DIR}')
print(f'공통 Fold: {FOLD_CSV}')
print(f'초기 adapter: {SOURCE_ADAPTER_DIR}')
print(f'24-class 실험 출력: {RUN_DIR}')

### 2. Qwen3-VL LoRA 패키지 설치

Colab의 PyTorch와 CUDA는 교체하지 않고 Transformers, PEFT, bitsandbytes, Qwen 이미지 유틸리티만 설치합니다. 버전을 고정해 다음 실행에서도 같은 API를 사용합니다. 설치 후 import 문제가 발생할 때만 런타임을 재시작하고 1번 셀부터 다시 실행합니다.

In [ ]:
%pip install -q transformers==5.13.0 accelerate==1.14.0 peft==0.19.1 bitsandbytes qwen-vl-utils==0.0.14 ipywidgets
%pip uninstall -y torchao

### 3. 공통 Fold 0과 Train 데이터 무결성 감사

저장된 Fold를 다시 계산하지 않고 `Id`로 Train과 병합합니다. 9,535개 ID가 정확히 한 번씩 존재하고 Fold별 개수가 기존 결과와 같은지 검사합니다. 여기서 통과해야 이후 Validation 점수를 팀원과 비교할 수 있습니다.

In [ ]:
import ast
import hashlib
import json
import math
import random
import re
import shutil
from itertools import permutations

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

SEED = 42
VALIDATION_FOLD = 0
EXPECTED_FOLD_COUNTS = {0: 1915, 1: 1905, 2: 1886, 3: 1911, 4: 1918}
EXPECTED_TRAIN_COUNT = 7620
EXPECTED_VALIDATION_COUNT = 1915

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

train = pd.read_csv(TRAIN_CSV, dtype=str, encoding='utf-8-sig')
folds = pd.read_csv(FOLD_CSV, encoding='utf-8-sig')
assert list(folds.columns) == ['Id', 'fold'], folds.columns.tolist()
assert len(train) == len(folds) == 9535
assert train['Id'].is_unique and folds['Id'].is_unique
assert folds['Id'].notna().all() and folds['fold'].notna().all()
folds['fold'] = folds['fold'].astype(int)
assert folds['fold'].value_counts().sort_index().to_dict() == EXPECTED_FOLD_COUNTS

train = train.merge(folds, on='Id', how='left', validate='one_to_one')
assert train['fold'].notna().all()
train_fold0 = train[train['fold'] != VALIDATION_FOLD].reset_index(drop=True)
validation_fold0 = train[train['fold'] == VALIDATION_FOLD].reset_index(drop=True)
assert len(train_fold0) == EXPECTED_TRAIN_COUNT
assert len(validation_fold0) == EXPECTED_VALIDATION_COUNT
assert set(train_fold0['Id']).isdisjoint(set(validation_fold0['Id']))

def parse_answer(value):
    answer = ast.literal_eval(value) if isinstance(value, str) else list(value)
    assert isinstance(answer, list) and len(answer) == 4
    assert sorted(answer) == [1, 2, 3, 4]
    return [int(number) for number in answer]

assert train['Answer'].apply(parse_answer).map(tuple).nunique() == 24
fold_sha256 = hashlib.sha256(FOLD_CSV.read_bytes()).hexdigest()
print(f'Fold SHA-256: {fold_sha256}')
print(f'Fold 0 Train: {len(train_fold0):,}')
print(f'Fold 0 Validation: {len(validation_fold0):,}')
print('공통 Fold 감사 통과')

### 4. Train 전용 Answer 분포 보존 증강 생성

각 Train ID에 비동일 shuffled copy 한 건을 만듭니다. 다만 임의 순열을 고르는 대신, 증강 후 24개 `Answer` 클래스의 개수가 원본과 정확히 같도록 목표 Answer를 먼저 배정합니다. 이미지 파일은 복사하지 않고 Input 파일명 순서만 바꿉니다.

이 셀은 모든 증강이 실제 비동일 순열인지, 이미지 재배열과 정답 변환이 일치하는지, 원본과 증강의 Answer 분포 차이가 0인지 검사합니다. Validation 1,915개는 변경하지 않습니다.

In [ ]:
IDENTITY_ORDER = (1, 2, 3, 4)
source_answers = [tuple(parse_answer(value)) for value in train_fold0['Answer']]
sorted_indices = sorted(
    range(len(train_fold0)),
    key=lambda index: (source_answers[index], train_fold0.iloc[index]['Id']),
)
sorted_answers = [source_answers[index] for index in sorted_indices]
max_class_count = max(pd.Series(source_answers).value_counts())
distribution_shift = int(max_class_count)
shifted_answers = sorted_answers[distribution_shift:] + sorted_answers[:distribution_shift]
assert all(source != target for source, target in zip(sorted_answers, shifted_answers))

target_answer_by_index = [None] * len(train_fold0)
for sorted_position, row_index in enumerate(sorted_indices):
    target_answer_by_index[row_index] = shifted_answers[sorted_position]
assert all(target is not None for target in target_answer_by_index)

def augment_row(row, target_answer):
    result = row.copy()
    old_inputs = [row[f'Input_{index}'] for index in IDENTITY_ORDER]
    old_answer = parse_answer(row['Answer'])
    target_answer = list(target_answer)
    order = tuple(old_answer.index(position) + 1 for position in target_answer)
    assert order != IDENTITY_ORDER
    for new_index, old_index in enumerate(order, start=1):
        result[f'Input_{new_index}'] = old_inputs[old_index - 1]
    new_answer = [old_answer[old_index - 1] for old_index in order]
    assert new_answer == target_answer
    result['Answer'] = str(new_answer)
    result['_variant'] = 'distribution_preserving_augmented'
    result['_input_order'] = str(list(order))
    return result

original_train = train_fold0.copy()
original_train['_variant'] = 'original'
original_train['_input_order'] = str(list(IDENTITY_ORDER))
augmented_train = pd.DataFrame([
    augment_row(row, target_answer_by_index[index])
    for index, (_, row) in enumerate(train_fold0.iterrows())
])
train_examples = pd.concat([original_train, augmented_train], ignore_index=True)
validation_examples = validation_fold0.copy()
validation_examples['_variant'] = 'original'
validation_examples['_input_order'] = str(list(IDENTITY_ORDER))

assert len(train_examples) == 15240
assert train_examples['_variant'].value_counts().to_dict() == {
    'original': 7620, 'distribution_preserving_augmented': 7620
}
assert len(validation_examples) == 1915
assert (validation_examples['_variant'] == 'original').all()
original_distribution = original_train['Answer'].apply(parse_answer).map(tuple).value_counts().sort_index()
augmented_distribution = augmented_train['Answer'].apply(parse_answer).map(tuple).value_counts().sort_index()
assert original_distribution.equals(augmented_distribution)
assert augmented_distribution.size == 24
assert all(order != str(list(IDENTITY_ORDER)) for order in augmented_train['_input_order'])

audit_index = 0
print('원본 Input:', [original_train.iloc[audit_index][f'Input_{i}'] for i in IDENTITY_ORDER])
print('원본 Answer:', original_train.iloc[audit_index]['Answer'])
print('증강 Input 순서:', augmented_train.iloc[audit_index]['_input_order'])
print('증강 Input:', [augmented_train.iloc[audit_index][f'Input_{i}'] for i in IDENTITY_ORDER])
print('증강 Answer:', augmented_train.iloc[audit_index]['Answer'])
print(f'Answer classes: {augmented_distribution.size}')
print(f'원본-증강 클래스별 최대 개수 차이: {(original_distribution - augmented_distribution).abs().max()}')
print(f'분포 보존 순환 이동 크기: {distribution_shift:,}')
print(f'Train examples: {len(train_examples):,}')
print(f'Validation examples: {len(validation_examples):,}')

### 5. 조건부 프롬프트와 24개 chronological class 정의

기존 모델과 동일한 조건부 프롬프트를 사용합니다. 4!개의 chronological order를 0~23번 class로 고정하고, 평가 직전에 공식 Answer 형식으로 역변환합니다. 분류 입력에는 정답이나 assistant 응답을 포함하지 않습니다.

In [ ]:
CLAUSE_PATTERN = re.compile(
    r'\b(then|after which|after that|and finally|finally|next|before that|followed by|first|before|after)\b,?',
    flags=re.IGNORECASE,
)
STRUCTURED_HINT = (
    'This caption describes distinct sequential stages, often marked by words like '
    "'then', 'after which', or 'finally'. Match each stage of the story to its corresponding image."
)
CONTINUOUS_HINT = (
    'This caption describes a single continuous action or a brief moment. '
    'Focus on subtle visual cues such as camera movement direction, object position changes, '
    'hand/body motion, and lighting changes to infer the temporal order.'
)

def split_clauses(sentence, max_clauses=4):
    sentence = sentence.strip()
    parts = CLAUSE_PATTERN.split(sentence)
    clauses = []
    current = parts[0].strip(' ,;.')
    for index in range(1, len(parts), 2):
        if current:
            clauses.append(current)
        current = parts[index + 1].strip(' ,;.')
    if current:
        clauses.append(current)
    clauses = [clause for clause in clauses if clause]
    if len(clauses) > max_clauses:
        clauses = clauses[:max_clauses - 1] + [' '.join(clauses[max_clauses - 1:])]
    return clauses or [sentence]

def answer_to_chronological_order(answer):
    answer = parse_answer(answer)
    return [index for _, index in sorted((position, index) for index, position in enumerate(answer, 1))]

def chronological_order_to_answer(order):
    assert isinstance(order, list) and sorted(order) == [1, 2, 3, 4]
    answer = [0, 0, 0, 0]
    for position, image_number in enumerate(order, start=1):
        answer[image_number - 1] = position
    return answer

def build_prompt_messages(row, image_dir):
    sentence = row['Sentence']
    hint = STRUCTURED_HINT if len(split_clauses(sentence)) >= 3 else CONTINUOUS_HINT
    content = []
    for image_number in IDENTITY_ORDER:
        image_path = image_dir / row['Id'] / row[f'Input_{image_number}']
        assert image_path.is_file(), image_path
        content.append({'type': 'image', 'image': str(image_path)})
        content.append({'type': 'text', 'text': f'\nImage {image_number}\n'})
    content.append({
        'type': 'text',
        'text': (
            f'Thinking about the sentence: \"{sentence}\"\n'
            f'{hint}\n'
            'Look at the 4 images above labeled Image 1 to Image 4. '
            'Determine the correct chronological order of these images to match the sentence. '
            'Provide the answer ONLY as a Python list of integers. '
            'Example: [1, 2, 3, 4]'
        ),
    })
    return [{'role': 'user', 'content': content}]

CHRONOLOGICAL_CLASSES = tuple(permutations(IDENTITY_ORDER))
CHRONOLOGICAL_TO_CLASS = {order: index for index, order in enumerate(CHRONOLOGICAL_CLASSES)}
assert len(CHRONOLOGICAL_CLASSES) == len(CHRONOLOGICAL_TO_CLASS) == 24

def answer_to_class_index(answer):
    order = tuple(answer_to_chronological_order(answer))
    return CHRONOLOGICAL_TO_CLASS[order]

def class_index_to_answer(class_index):
    return chronological_order_to_answer(list(CHRONOLOGICAL_CLASSES[int(class_index)]))

for answer in permutations(IDENTITY_ORDER):
    answer = list(answer)
    assert chronological_order_to_answer(answer_to_chronological_order(answer)) == answer
    assert class_index_to_answer(answer_to_class_index(answer)) == answer
print('24개 Answer ↔ chronological class 왕복 변환 통과')

### 6. Prompt-only Dataset과 24-class batch 정의

각 입력에는 네 이미지와 Caption·조건부 힌트만 포함합니다. 정답 assistant 문자열은 모델 입력에 넣지 않고 별도의 class label로만 전달하므로 정답 토큰 누수가 없습니다.

In [ ]:
from dataclasses import dataclass
from torch.utils.data import DataLoader, Dataset
from qwen_vl_utils import process_vision_info

class OrderingClassificationDataset(Dataset):
    def __init__(self, dataframe, image_dir):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = image_dir

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        return {
            'Id': row['Id'],
            'messages': build_prompt_messages(row, self.image_dir),
            'Answer': row['Answer'],
            'class_label': answer_to_class_index(row['Answer']),
        }

@dataclass
class ClassificationCollator:
    processor: object

    def __call__(self, examples):
        texts = []
        all_images = []
        for example in examples:
            messages = example['messages']
            assert len(messages) == 1
            texts.append(self.processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            ))
            image_inputs, video_inputs = process_vision_info(messages)
            assert image_inputs is not None and len(image_inputs) == 4
            assert not video_inputs
            all_images.extend(image_inputs)
        batch = self.processor(
            text=texts, images=all_images, padding=True, return_tensors='pt'
        )
        batch['class_labels'] = torch.tensor(
            [example['class_label'] for example in examples], dtype=torch.long
        )
        batch['sample_ids'] = [example['Id'] for example in examples]
        batch['target_answers'] = [example['Answer'] for example in examples]
        return dict(batch)

training_dataset = OrderingClassificationDataset(train_examples, TRAIN_IMAGE_DIR)
validation_dataset = OrderingClassificationDataset(validation_examples, TRAIN_IMAGE_DIR)
assert len(training_dataset) == 15240
assert len(validation_dataset) == 1915
print(f'24-class Train Dataset: {len(training_dataset):,}')
print(f'Validation Dataset: {len(validation_dataset):,}')

### 7. 기존 LoRA와 24-class head 로드

이전 분포 보존 `final_adapter`의 7개 LoRA 가중치를 학습 가능한 상태로 불러옵니다. Prompt 마지막 토큰의 멀티모달 hidden state를 입력으로 받는 24-class Linear head를 새로 추가합니다.

In [ ]:
import transformers
import peft
import torch.nn as nn
from transformers import AutoModelForImageTextToText, AutoProcessor
from peft import PeftModel
from safetensors.torch import load_file, save_file

LORA_TARGET_MODULES = [
    'q_proj', 'k_proj', 'v_proj', 'o_proj',
    'gate_proj', 'up_proj', 'down_proj',
]
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

print(f'transformers: {transformers.__version__}')
print(f'peft: {peft.__version__}')
processor = AutoProcessor.from_pretrained(MODEL_NAME)
processor.tokenizer.padding_side = 'right'

base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    device_map='auto',
    low_cpu_mem_usage=True,
)
final_artifacts = [FINAL_ADAPTER_DIR, CLASSIFIER_PATH, CLASS_MAPPING_PATH]
final_complete = all(path.exists() for path in final_artifacts)
if any(path.exists() for path in final_artifacts) and not final_complete:
    raise RuntimeError(f'24-class 결과 폴더가 불완전합니다: {FINAL_ADAPTER_DIR}')

adapter_to_load = FINAL_ADAPTER_DIR if final_complete else SOURCE_ADAPTER_DIR
model = PeftModel.from_pretrained(
    base_model, adapter_to_load, is_trainable=not final_complete
)
TRAINING_REQUIRED = not final_complete
loaded_lora = model.peft_config['default']
assert set(loaded_lora.target_modules) == set(LORA_TARGET_MODULES)
assert loaded_lora.r == LORA_R

base_config = model.get_base_model().config
text_config = getattr(base_config, 'text_config', base_config)
HIDDEN_SIZE = int(text_config.hidden_size)
CLASS_COUNT = len(CHRONOLOGICAL_CLASSES)
model_device = next(model.parameters()).device
classifier = nn.Linear(HIDDEN_SIZE, CLASS_COUNT, bias=True).to(
    device=model_device, dtype=torch.float32
)
if final_complete:
    classifier.load_state_dict(load_file(str(CLASSIFIER_PATH), device='cpu'))
    classifier.to(model_device)
    classifier.eval()
    print('저장된 24-class adapter와 head를 불러왔습니다. 학습을 건너뜁니다.')
else:
    print(f'기존 생성 모델에서 24-class 학습을 시작합니다: {SOURCE_ADAPTER_DIR}')

model.config.use_cache = False
if TRAINING_REQUIRED:
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    model.enable_input_require_grads()
model.print_trainable_parameters()
print('LoRA 대상:', LORA_TARGET_MODULES)
print(f'Classification head: {HIDDEN_SIZE:,} → {CLASS_COUNT}')

### 8. 첫 배치 hidden state와 class loss 감사

Qwen의 LM head를 거치지 않고 멀티모달 core의 마지막 prompt hidden state만 추출합니다. 실제 첫 배치에서 hidden 크기·24개 logits·유한한 cross-entropy loss를 확인하며 backward 학습은 하지 않습니다.

In [ ]:
import torch.nn.functional as F

collator = ClassificationCollator(processor)
multimodal_core = model.get_base_model().model
assert hasattr(multimodal_core, 'forward')

def move_classification_batch(batch, device):
    class_labels = batch.pop('class_labels').to(device, non_blocking=True)
    sample_ids = batch.pop('sample_ids')
    target_answers = batch.pop('target_answers')
    model_inputs = {
        key: value.to(device, non_blocking=True) if torch.is_tensor(value) else value
        for key, value in batch.items()
    }
    return model_inputs, class_labels, sample_ids, target_answers

def pool_last_prompt_token(last_hidden_state, attention_mask):
    last_indices = attention_mask.sum(dim=1).long() - 1
    batch_indices = torch.arange(last_hidden_state.shape[0], device=last_hidden_state.device)
    return last_hidden_state[batch_indices, last_indices].float()

audit_batch = collator([training_dataset[0]])
audit_inputs, audit_labels, _, _ = move_classification_batch(audit_batch, model_device)
model.eval()
classifier.eval()
with torch.inference_mode(), torch.autocast(device_type='cuda', dtype=torch.bfloat16):
    audit_outputs = multimodal_core(**audit_inputs, use_cache=False, return_dict=True)
audit_pooled = pool_last_prompt_token(
    audit_outputs.last_hidden_state, audit_inputs['attention_mask']
)
with torch.autocast(device_type='cuda', enabled=False):
    audit_logits = classifier(audit_pooled)
    audit_loss = F.cross_entropy(audit_logits, audit_labels)
assert audit_pooled.shape == (1, HIDDEN_SIZE)
assert audit_logits.shape == (1, CLASS_COUNT)
assert torch.isfinite(audit_loss)
print(f'Prompt hidden state: {tuple(audit_pooled.shape)}')
print(f'Class logits: {tuple(audit_logits.shape)}')
print(f'초기 class loss: {audit_loss.item():.6f}')
print('24-class 입력·loss 감사 통과')
del audit_batch, audit_inputs, audit_outputs, audit_pooled, audit_logits, audit_loss
torch.cuda.empty_cache()

### 9. 24-class head와 LoRA 공동 학습

원본 7,620개와 분포 보존 증강 7,620개를 한 번씩 학습합니다. 기존 LoRA에는 작은 learning rate `5e-5`, 새 classification head에는 `5e-4`를 적용합니다.

24개 클래스 불균형은 완전한 역빈도가 아니라 제곱근 역빈도 가중치로 완화합니다. 학습 완료 후 LoRA adapter·classification head·class mapping을 같은 `final_adapter/` 폴더에 저장합니다.

In [ ]:
import time
import bitsandbytes as bnb

BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LORA_LEARNING_RATE = 5e-5
CLASSIFIER_LEARNING_RATE = 5e-4
CLASSIFIER_WEIGHT_DECAY = 0.01
CLASSIFIER_ONLY_WARMUP_STEPS = 200
DATALOADER_NUM_WORKERS = 2
DATALOADER_PREFETCH_FACTOR = 2
LOG_EVERY_OPTIMIZER_STEPS = 10

training_class_labels = np.array([
    answer_to_class_index(answer) for answer in train_examples['Answer']
], dtype=np.int64)
class_counts = np.bincount(training_class_labels, minlength=CLASS_COUNT)
assert len(class_counts) == CLASS_COUNT and np.all(class_counts > 0)
class_weight_values = np.sqrt(class_counts.max() / class_counts)
class_weight_values = class_weight_values / class_weight_values.mean()
class_weights = torch.tensor(
    class_weight_values, dtype=torch.float32, device=model_device
)
print(f'24-class count range: {class_counts.min():,} ~ {class_counts.max():,}')
print(f'Class weight range: {class_weight_values.min():.4f} ~ {class_weight_values.max():.4f}')

run_config = {
    'scope': 'train_fold0_only_no_test_access',
    'fold_sha256': fold_sha256,
    'validation_fold': VALIDATION_FOLD,
    'unique_train_samples': len(train_fold0),
    'training_examples': len(train_examples),
    'validation_samples': len(validation_examples),
    'training_engine': 'prompt_hidden_state_24_class_manual_loop',
    'source_adapter': str(SOURCE_ADAPTER_DIR),
    'augmentation': 'exact_24_class_distribution_preserving_non_identity_permutation',
    'augmentation_distribution_shift': distribution_shift,
    'model_name': MODEL_NAME,
    'objective': 'sqrt_inverse_frequency_weighted_cross_entropy_24_classes',
    'class_semantics': 'chronological_order_then_convert_to_official_answer',
    'lora_learning_rate': LORA_LEARNING_RATE,
    'classifier_learning_rate': CLASSIFIER_LEARNING_RATE,
    'classifier_only_warmup_optimizer_steps': CLASSIFIER_ONLY_WARMUP_STEPS,
    'learning_rate_schedule': 'constant_one_epoch',
    'effective_batch_size': BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
    'dataloader': {
        'num_workers': DATALOADER_NUM_WORKERS,
        'prefetch_factor': DATALOADER_PREFETCH_FACTOR,
        'persistent_workers': True,
    },
    'lora': {
        'r': LORA_R, 'alpha': LORA_ALPHA, 'dropout': LORA_DROPOUT,
        'target_modules': LORA_TARGET_MODULES,
    },
}
with (RUN_DIR / 'run_config.json').open('w', encoding='utf-8') as file:
    json.dump(run_config, file, ensure_ascii=False, indent=2)

summary_path = RUN_DIR / 'training_summary.json'
log_path = RUN_DIR / 'training_log.csv'
log_records = []

if TRAINING_REQUIRED:
    torch.backends.cuda.matmul.allow_tf32 = True
    lora_parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
    classifier_parameters = list(classifier.parameters())
    assert lora_parameters and classifier_parameters
    trainable_parameters = lora_parameters + classifier_parameters
    training_device = lora_parameters[0].device
    assert training_device == model_device
    optimizer = bnb.optim.AdamW8bit([
        {
            'params': lora_parameters,
            'lr': 0.0,
            'weight_decay': 0.0,
        },
        {
            'params': classifier_parameters,
            'lr': CLASSIFIER_LEARNING_RATE,
            'weight_decay': CLASSIFIER_WEIGHT_DECAY,
        },
    ])
    optimizer.zero_grad(set_to_none=True)
    global_optimizer_step = 0

    generator = torch.Generator()
    generator.manual_seed(SEED)
    loader = DataLoader(
        training_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collator,
        num_workers=DATALOADER_NUM_WORKERS,
        prefetch_factor=DATALOADER_PREFETCH_FACTOR,
        persistent_workers=True,
        pin_memory=True,
        generator=generator,
    )
    model.train()
    classifier.train()
    model.config.use_cache = False
    loss_sum = 0.0
    correct_sum = 0
    training_started = time.perf_counter()
    progress = tqdm(loader, desc='cls24_train')

    for micro_step, batch in enumerate(progress, start=1):
        model_inputs, class_labels, _, _ = move_classification_batch(
            batch, training_device
        )
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            outputs = multimodal_core(
                **model_inputs, use_cache=False, return_dict=True
            )
        pooled = pool_last_prompt_token(
            outputs.last_hidden_state, model_inputs['attention_mask']
        )
        with torch.autocast(device_type='cuda', enabled=False):
            logits = classifier(pooled)
            raw_loss = F.cross_entropy(logits, class_labels, weight=class_weights)
        assert torch.isfinite(raw_loss), f'비정상 loss: {raw_loss.item()}'
        (raw_loss / GRADIENT_ACCUMULATION_STEPS).backward()
        loss_sum += float(raw_loss.detach().item())
        correct_sum += int((logits.detach().argmax(dim=-1) == class_labels).sum().item())

        should_update = (
            micro_step % GRADIENT_ACCUMULATION_STEPS == 0
            or micro_step == len(loader)
        )
        if should_update:
            if global_optimizer_step < CLASSIFIER_ONLY_WARMUP_STEPS:
                for parameter in lora_parameters:
                    parameter.grad = None
            grad_norm = torch.nn.utils.clip_grad_norm_(trainable_parameters, max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            global_optimizer_step += 1
            if global_optimizer_step == CLASSIFIER_ONLY_WARMUP_STEPS:
                optimizer.param_groups[0]['lr'] = LORA_LEARNING_RATE
                print(f'LoRA 공동 학습 시작: optimizer step {global_optimizer_step + 1}')
            log_records.append({
                'micro_step': micro_step,
                'optimizer_step': global_optimizer_step,
                'loss': float(raw_loss.detach().item()),
                'running_accuracy': correct_sum / micro_step,
                'grad_norm': float(grad_norm),
                'lora_learning_rate': optimizer.param_groups[0]['lr'],
                'classifier_learning_rate': CLASSIFIER_LEARNING_RATE,
            })
            if global_optimizer_step == 1 or global_optimizer_step % LOG_EVERY_OPTIMIZER_STEPS == 0:
                progress.set_postfix(
                    loss=f'{raw_loss.item():.4f}',
                    acc=f'{correct_sum / micro_step:.4f}',
                    optimizer_step=global_optimizer_step,
                )

    elapsed_seconds = time.perf_counter() - training_started
    FINAL_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(FINAL_ADAPTER_DIR, safe_serialization=True)
    processor.save_pretrained(FINAL_ADAPTER_DIR)
    classifier_state = {
        key: value.detach().cpu().contiguous()
        for key, value in classifier.state_dict().items()
    }
    save_file(classifier_state, str(CLASSIFIER_PATH))
    class_mapping = {
        'class_semantics': 'chronological_order',
        'class_index_to_chronological_order': {
            str(index): list(order)
            for index, order in enumerate(CHRONOLOGICAL_CLASSES)
        },
    }
    with CLASS_MAPPING_PATH.open('w', encoding='utf-8') as file:
        json.dump(class_mapping, file, ensure_ascii=False, indent=2)
    pd.DataFrame(log_records).to_csv(log_path, index=False, encoding='utf-8-sig')
    training_summary = {
        'training_engine': 'prompt_hidden_state_24_class_manual_loop',
        'sample_count': len(training_dataset),
        'mean_training_loss': loss_sum / len(loader),
        'training_accuracy': correct_sum / len(training_dataset),
        'elapsed_seconds': elapsed_seconds,
        'source_adapter': str(SOURCE_ADAPTER_DIR),
        'final_adapter_dir': str(FINAL_ADAPTER_DIR),
        'classifier_path': str(CLASSIFIER_PATH),
        'exact_match_evaluated': False,
    }
    with summary_path.open('w', encoding='utf-8') as file:
        json.dump(training_summary, file, ensure_ascii=False, indent=2)

    model.gradient_checkpointing_disable()
    model.config.use_cache = True
    print(f'24-class 학습 완료: {FINAL_ADAPTER_DIR}')
    print(f'Classification head 저장: {CLASSIFIER_PATH}')
else:
    print(f'기존 24-class 모델 사용: {FINAL_ADAPTER_DIR}')

### 10. 전체 Fold 0 24-class Validation 평가

Fold 0의 원본 1,915개에서 한 번의 forward로 24개 class logits를 계산합니다. 가장 높은 chronological class를 공식 Answer로 변환하므로 모든 출력은 항상 유효한 순열입니다.

In [ ]:
def compute_ordering_metrics(targets, predictions):
    targets = np.asarray(targets, dtype=int)
    predictions = np.asarray(predictions, dtype=int)
    exact = float(np.mean(np.all(targets == predictions, axis=1)))
    position = float(np.mean(targets == predictions))
    pairwise_values = []
    for target, prediction in zip(targets, predictions):
        correct = 0
        total = 0
        for first in range(4):
            for second in range(first + 1, 4):
                correct += (prediction[first] < prediction[second]) == (target[first] < target[second])
                total += 1
        pairwise_values.append(correct / total)
    pairwise = float(np.mean(pairwise_values))
    return {
        'exact_match_accuracy': exact,
        'position_accuracy': position,
        'pairwise_ordering_accuracy': pairwise,
        'mean_kendall_tau': 2.0 * pairwise - 1.0,
    }

model.config.use_cache = False
model.eval()
classifier.eval()
processor.tokenizer.padding_side = 'right'
EVAL_BATCH_SIZE = 4
results = []

validation_loader = DataLoader(
    validation_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collator,
    num_workers=DATALOADER_NUM_WORKERS,
    prefetch_factor=DATALOADER_PREFETCH_FACTOR,
    persistent_workers=True,
    pin_memory=True,
)

for batch in tqdm(validation_loader, desc='Fold 0 cls24 Validation'):
    model_inputs, class_labels, sample_ids, target_answers = move_classification_batch(
        batch, model_device
    )
    with torch.inference_mode(), torch.autocast(
        device_type='cuda', dtype=torch.bfloat16
    ):
        outputs = multimodal_core(
            **model_inputs, use_cache=False, return_dict=True
        )
    pooled = pool_last_prompt_token(
        outputs.last_hidden_state, model_inputs['attention_mask']
    )
    with torch.inference_mode(), torch.autocast(device_type='cuda', enabled=False):
        logits = classifier(pooled)
        probabilities = torch.softmax(logits, dim=-1)
    predicted_classes = probabilities.argmax(dim=-1)
    confidences = probabilities.max(dim=-1).values
    for sample_id, target_text, target_class, predicted_class, confidence in zip(
        sample_ids, target_answers, class_labels, predicted_classes, confidences
    ):
        target_answer = parse_answer(target_text)
        prediction = class_index_to_answer(int(predicted_class.item()))
        results.append({
            'Id': sample_id,
            'target_answer': str(target_answer),
            'predicted_answer': str(prediction),
            'target_class': int(target_class.item()),
            'predicted_class': int(predicted_class.item()),
            'confidence': float(confidence.item()),
        })

assert len(results) == EXPECTED_VALIDATION_COUNT
targets = [ast.literal_eval(row['target_answer']) for row in results]
predictions = [ast.literal_eval(row['predicted_answer']) for row in results]
metrics = compute_ordering_metrics(targets, predictions)
identity_mask = np.array([target == [1, 2, 3, 4] for target in targets])
shuffled_exact = float(np.mean(
    np.all(np.asarray(targets)[~identity_mask] == np.asarray(predictions)[~identity_mask], axis=1)
))

validation_predictions_path = RUN_DIR / 'fold0_validation_predictions.csv'
validation_summary_path = RUN_DIR / 'fold0_validation_summary.json'
pd.DataFrame(results).to_csv(validation_predictions_path, index=False, encoding='utf-8-sig')
validation_summary = {
    'scope': 'train_fold0_validation_only_no_test_access',
    'sample_count': len(results),
    'fold_sha256': fold_sha256,
    'metrics': metrics,
    'shuffled_only_exact_match_accuracy': shuffled_exact,
    'identity_only_exact_match_accuracy': float(np.mean(
        np.all(np.asarray(targets)[identity_mask] == np.asarray(predictions)[identity_mask], axis=1)
    )),
    'valid_permutation_rate': 1.0,
    'adapter': str(FINAL_ADAPTER_DIR),
    'classifier': str(CLASSIFIER_PATH),
}
with validation_summary_path.open('w', encoding='utf-8') as file:
    json.dump(validation_summary, file, ensure_ascii=False, indent=2)
if summary_path.is_file():
    with summary_path.open(encoding='utf-8') as file:
        training_summary = json.load(file)
    training_summary['exact_match_evaluated'] = True
    training_summary['validation_metrics'] = metrics
    with summary_path.open('w', encoding='utf-8') as file:
        json.dump(training_summary, file, ensure_ascii=False, indent=2)

print(json.dumps(validation_summary, ensure_ascii=False, indent=2))
print(f'예측 저장: {validation_predictions_path}')
print(f'요약 저장: {validation_summary_path}')

### 11. 클래스별 성능과 목표 차이 진단

24개 chronological class별 정확도와 평균 confidence를 저장합니다. 현재 생성형 기준선 0.5253과 비교하고, Exact Match 0.60까지 필요한 추가 정답 수를 계산합니다.

In [ ]:
result_frame = pd.DataFrame(results)
result_frame['correct'] = (
    result_frame['target_class'] == result_frame['predicted_class']
)
per_class = result_frame.groupby('target_class', as_index=False).agg(
    sample_count=('Id', 'size'),
    correct_count=('correct', 'sum'),
    exact_match=('correct', 'mean'),
    mean_confidence=('confidence', 'mean'),
)
per_class['chronological_order'] = per_class['target_class'].map(
    lambda index: str(list(CHRONOLOGICAL_CLASSES[int(index)]))
)
per_class = per_class[[
    'target_class', 'chronological_order', 'sample_count',
    'correct_count', 'exact_match', 'mean_confidence',
]].sort_values(['exact_match', 'sample_count'], ascending=[True, False])
per_class_path = RUN_DIR / 'fold0_per_class_accuracy.csv'
per_class.to_csv(per_class_path, index=False, encoding='utf-8-sig')

GENERATION_BASELINE_EXACT = 0.5253263707571801
TARGET_EXACT = 0.60
current_exact = metrics['exact_match_accuracy']
current_correct = int(result_frame['correct'].sum())
target_correct = math.floor(TARGET_EXACT * len(result_frame)) + 1
additional_correct_needed = max(0, target_correct - current_correct)
comparison = {
    'generation_baseline_exact_match': GENERATION_BASELINE_EXACT,
    'classification_exact_match': current_exact,
    'difference_vs_generation': current_exact - GENERATION_BASELINE_EXACT,
    'target_strictly_over_0.60_correct_count': target_correct,
    'additional_correct_needed': additional_correct_needed,
}
print(json.dumps(comparison, ensure_ascii=False, indent=2))
print(f'클래스별 결과 저장: {per_class_path}')
display(per_class)